In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import datetime 
import plotly
import plotly.graph_objs as go
from plotly.offline import plot
from plotly import tools
import re

# plotly.io.orca.config.executable  = 'C:/Users/wg984/AppData/Local/conda/conda/envs/plotly3/orca_app/orca.exe'

In [6]:
# called consent date enrolled date, floor 1 date AM in redcap
# consent = pd.read_csv('//mad3/MGH-NEURO-CDAC/Projects/ICU_SLEEP_STUDY/data/data_analysis/ExportedReports/ICUSLEEP-ConsentDateEnrolledD_DATA_LABELS.csv')
consent = pd.read_csv('C:/Users/wg984/Downloads/ICUSLEEP-ConsentDateEnrolledD_DATA_LABELS.csv')

consent.columns = consent.columns.str.strip()
consent.columns = ['Study ID:', 'Event Name', 'Repeat Instrument', 'Repeat Instance',
       'Consenting Date & Time', 'Evaluation Date and Time', 'First Day Enrolled in Study:']
consent.head()

,Study ID:,Event Name,Repeat Instrument,Repeat Instance,Consenting Date & Time,Evaluation Date and Time,First Day Enrolled in Study:
0,1,"ICU Day 1, am Screenng/Enroll",NaN,NaN,2018-06-06 10:00,2018-06-06 15:15,2018-06-06
1,1,"ICU Day 1, pm",NaN,NaN,NaN,2018-06-06 16:51,NaN
2,1,"Floor Day 1, pm",NaN,NaN,NaN,2018-06-08 17:19,NaN
3,2,"ICU Day 1, am Screenng/Enroll",NaN,NaN,2018-06-11 09:56,NaN,2018-06-11
4,2,"ICU Day 1, pm",NaN,NaN,NaN,2018-06-11 20:14,NaN


In [7]:
consent.columns

Index(['Study ID:', 'Event Name', 'Repeat Instrument', 'Repeat Instance',
       'Consenting Date & Time', 'Evaluation Date and Time',
       'First Day Enrolled in Study:'],
      dtype='object')

In [8]:
enrollment_dates = pd.DataFrame(columns=['date'])
for study_id in np.unique(consent['Study ID:']):
    consent_id = consent.loc[consent['Study ID:'] == study_id,:]
    all_dates = consent_id['Consenting Date & Time'].dropna().values.tolist() + consent_id['Evaluation Date and Time'].dropna().values.tolist() + consent_id['First Day Enrolled in Study:'].dropna().values.tolist()
    if len(all_dates) == 0: continue
    date_id = pd.to_datetime(all_dates[0], infer_datetime_format=1) # .date()
    enrollment_dates.loc[study_id, 'date'] = date_id
    
enrollment_dates.drop('98a', inplace=True)

In [9]:
enrollment_dates

,date
1,2018-06-06 10:00:00
10,2018-09-20 10:46:00
100,2019-06-20 12:45:00
101,2019-06-25 13:23:00
102,2019-06-25 13:48:00
...,...
95,2019-06-06 09:32:00
96,2019-06-12 12:45:00
97,2019-06-12 13:00:00
98,2019-06-24 12:36:00


In [10]:
enrollment_dates = enrollment_dates.sort_values('date')
enrollment_dates['EnrollmentNo'] = np.arange(1,len(enrollment_dates)+1)
consent = enrollment_dates

In [12]:
not_in_study = ['22','26', '30', '42', '44', '53', '56', '70', '89', '109', '118', '123', '162']

In [13]:
start = consent['date'].iloc[0]
June2022 = pd.Timestamp('2022-06-01 10:00:00')
End2021 = pd.Timestamp('2022-01-01 10:00:00')
trace_End2021 = go.Scatter(x=[start,End2021], y=[0,750], name = '750 until end of 2021', hoverinfo = 'x+y')
trace_June2022 = go.Scatter(x=[start,June2022], y=[0,750], name = '750 until June 2022', hoverinfo = 'x+y')

In [14]:
currentDate = datetime.datetime.now()

In [15]:
# traceEnrollment = go.Scatter(x=consent['Consenting Date & Time'], y=consent['EnrollmentNo'], name = 'enrollment', hoverinfo = 'x+y')
traceEnrollment = go.Scatter(x=pd.concat([consent['date'],pd.Series(currentDate)]), y=pd.concat([consent['EnrollmentNo'],pd.Series(consent['EnrollmentNo'].iloc[-1])]), name = 'enrollment', hoverinfo = 'x+y')

In [16]:
currentEnrollment = consent.iloc[-1,:].EnrollmentNo

In [18]:
consent.tail()

,date,EnrollmentNo
290,2020-10-02 12:20:00,290
291,2020-10-02 12:25:00,291
293,2020-10-06 12:20:00,292
294,2020-10-06 12:30:00,293
292,2020-10-06 13:55:00,294


In [19]:
FutureDate = np.array([currentDate + datetime.timedelta(days = 7*iWeeks) for iWeeks in range(52*4)])
FutureEnrollment4perWeek = np.array([currentEnrollment + 4*iWeeks for iWeeks in range(52*4)])
FutureEnrollment5perWeek = np.array([currentEnrollment + 5*iWeeks for iWeeks in range(52*4)])


In [20]:
trace_4perWeek = go.Scatter(x=FutureDate[FutureEnrollment4perWeek<=754], y=FutureEnrollment4perWeek[FutureEnrollment4perWeek<=754], name = '4 per week', hoverinfo = 'x+y',
                         line = dict(
            dash = 'dashdot',
            width = 2,
            color = 'red'
        ),
                           )
trace_5perWeek = go.Scatter(x=FutureDate[FutureEnrollment5perWeek<=755], y=FutureEnrollment5perWeek[FutureEnrollment5perWeek<=755], name = '5 per week', hoverinfo = 'x+y',
                           line = dict(
            dash = 'dashdot',
            width = 2,
            color = 'green'
        ),
             )

trace_750subjects = go.Scatter(x=[start, pd.Timestamp('2022-11-01 10:00:00')], y=[750, 750], line = dict(
            width = 2,
            color = 'rgb(122, 122, 122)'
        ), name = 'enrollment goal = 750', hoverinfo = 'x+y' )


### compute weekly enrollment rate for last week and last month:

In [33]:
EnrollmentsLastWeek = np.sum((consent['date'] >= pd.Timestamp(currentDate.date() - datetime.timedelta(hours = 7*24))).values)
EnrollmentsLastMonth = np.sum((consent['date'] >= pd.Timestamp(currentDate.date() - datetime.timedelta(hours = 4*7*24))).values)
EnrollmentsLast3Months = np.sum((consent['date'] >= pd.Timestamp(currentDate.date() - datetime.timedelta(hours = 3*4*7*24))).values)
EnrollmentRateLastWeek = EnrollmentsLastWeek
EnrollmentRateLastMonth = EnrollmentsLastMonth/4
EnrollmentRateLast3Months = EnrollmentsLast3Months/12

FutureEnrollmentAsAverageLast4Weeks = np.array([currentEnrollment + EnrollmentRateLastMonth*iWeeks for iWeeks in range(52*4)])
FutureEnrollmentAsAverageLast12Weeks = np.array([currentEnrollment + EnrollmentRateLast3Months*iWeeks for iWeeks in range(52*4)])

plot_average = '12weeks'
if plot_average == '12weeks':
    trace_asAverage = go.Scatter(x=FutureDate[FutureEnrollmentAsAverageLast12Weeks<=755], y=FutureEnrollmentAsAverageLast12Weeks[FutureEnrollmentAsAverageLast12Weeks<=755], name = '%.1f per week (average last 12 weeks)'%EnrollmentRateLast3Months, hoverinfo = 'x+y',
                               line = dict(
                width = 2,
                color = 'orange'
            ),
                 )
elif plot_average == '4weeks':
    trace_asAverage = go.Scatter(x=FutureDate[FutureEnrollmentAsAverageLast4Weeks<=755], y=FutureEnrollmentAsAverageLast4Weeks[FutureEnrollmentAsAverageLast4Weeks<=755], name = '%.1f per week (average last 4 weeks)'%EnrollmentRateLastMonth, hoverinfo = 'x+y',
                               line = dict(
                width = 2,
                color = 'orange'
            ),
                 )

In [34]:
EnrollmentRateLast3Months

3.5833333333333335

In [35]:
fig = go.FigureWidget()

In [36]:
fig.add_trace(traceEnrollment);
fig.add_trace(trace_750subjects);
# fig.add_trace(trace_End2021)
# fig.add_trace(trace_June2022)
fig.add_trace(trace_4perWeek);
fig.add_trace(trace_5perWeek);
fig.add_trace(trace_asAverage);

In [37]:
weeks_average = {re.search('\d+', plot_average)[0]}

In [43]:
layout = go.Layout(
    showlegend=True,
    images = [dict(
        source= "https://upload.wikimedia.org/wikipedia/commons/d/d5/Emojione_1F525.svg",
        xref= "paper",
        yref= "paper",
        x= 0.61,
        y= 1.12,
        sizex= 0.1,
        sizey= 0.1,
        xanchor= "left",
        yanchor= "top"
      )],
    
    annotations=[
        
                dict(
            x=pd.Timestamp(currentDate + datetime.timedelta(hours = 24*10)),
            y=currentEnrollment+310,
            xref='x',
            yref='y',
            text= 'Enrollments last 7 days= \t '+ str(EnrollmentRateLastWeek),
            showarrow=False,
#             arrowhead=7,
            ax=500,
            ay=500
        ),
        
                dict(
            x=pd.Timestamp(currentDate + datetime.timedelta(hours = 24*10)),
            y=currentEnrollment+280,
            xref='x',
            yref='y',
            text= 'Weekly enrollment rate for last 4 weeks= \t' + str(EnrollmentRateLastMonth),
            showarrow=False,
#             arrowhead=7,
#             ax=500,
            ay=40
        ),
        
                dict(
            x=pd.Timestamp(currentDate + datetime.timedelta(hours = 24*10)),
            y=currentEnrollment+250,
            xref='x',
            yref='y',
            text= 'Weekly enrollment rate for last 12 weeks= \t %.1f'%EnrollmentRateLast3Months,
            showarrow=False,
#             arrowhead=7,
#             ax=500,
            ay=40
        ),
        
        
        dict(
            x=pd.Timestamp(currentDate),
            y=currentEnrollment,
            xref='x',
            yref='y',
            text= str(currentDate.date()) +' - ' + str(currentEnrollment) +' subjects enrolled',
            showarrow=True,
            arrowhead=7,
            ax=80,
            ay=40
        ),
                dict(
            x=FutureDate[FutureEnrollment5perWeek<=755][-1],
            y=750,
            xref='x',
            yref='y',
            text=str(FutureDate[FutureEnrollment5perWeek<=755][-1]).split(' ')[0],
            showarrow=True,
            arrowhead=7,
            ax=60,
            ay=-35
        ),
        
                dict(
            x=FutureDate[FutureEnrollment4perWeek<=754][-1],
            y=750,
            xref='x',
            yref='y',
            text=str(FutureDate[FutureEnrollment4perWeek<=754][-1]).split(' ')[0],
            showarrow=True,
            arrowhead=7,
            ax=60,
            ay=-35
        ),
        
                 dict(
            x=FutureDate[FutureEnrollmentAsAverageLast12Weeks<=754][-1],
            y=750,
            xref='x',
            yref='y',
            text=str(FutureDate[FutureEnrollmentAsAverageLast12Weeks<=754][-1]).split(' ')[0],
            showarrow=True,
            arrowhead=7,
            ax=60,
            ay=-10
        ),

#                  dict(
#             x=FutureDate[FutureEnrollmentAsAverageLast4Weeks<=754][-1],
#             y=750,
#             xref='x',
#             yref='y',
#             text=str(FutureDate[FutureEnrollmentAsAverageLast4Weeks<=754][-1]).split(' ')[0],
#             showarrow=True,
#             arrowhead=7,
#             ax=60,
#             ay=-10
#         ),
                         dict(
            x= 0.2, #consent.date.iloc[0],
            y=1.07,
            showarrow=False,
            xref='paper',
            yref='paper',
            text='# currently enrolled subjects with study ID: ' + str(currentEnrollment),

        ),
                                 dict(
            x= 0.2, #consent.date.iloc[0],
            y=1.05,
            showarrow=False,
            xref='paper',
            yref='paper',
            text='# subjects who will/might not be included in study: ' + str(len(not_in_study)),

        ),
                                 dict(
            x= 0.2, #consent.date.iloc[0],
            y=1.03,
            showarrow=False,
            xref='paper',
            yref='paper',
            text=f"Projected enrollment date for #750 based on last-12-week enrollment rate: " + str(FutureDate[FutureEnrollmentAsAverageLast12Weeks<=754][-1]).split(' ')[0],

        )
        
    ],
     title=('ICU-Sleep enrollment'),
)

In [44]:
fig.layout = layout

In [45]:
plot(fig, auto_open=True)

'temp-plot.html'

In [46]:
plot(fig,filename = 'C:/Users/wg984/Dropbox (Partners HealthCare)/ICU-SLEEP-MATERIALS/ICU_enrollment_burndownchart.html', auto_open=True) #  filename = 'MVP1_5_withSubplots.html'

'C:/Users/wg984/Dropbox (Partners HealthCare)/ICU-SLEEP-MATERIALS/ICU_enrollment_burndownchart.html'